# task_06 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_06/data/")


In [2]:

accounts = pd.read_csv(base / "accounts.csv")
invoices = pd.read_csv(base / "invoices.csv")
plan_history = pd.read_csv(base / "plan_history.csv")
tickets = pd.read_csv(base / "tickets.csv")


Q2: In Q2 2024, which segment has the highest average support ticket count per active account?

In [4]:
tickets["opened_date"] = pd.to_datetime(tickets["opened_date"])
q2_months = {"2024-04-01", "2024-05-01", "2024-06-01"}
active = invoices[invoices["month"].isin(q2_months)][["account_id"]].drop_duplicates().merge(accounts[["account_id", "segment"]], on="account_id")
q2_tickets = tickets[(tickets["opened_date"].dt.year == 2024) & (tickets["opened_date"].dt.month.isin([4, 5, 6]))]
ticket_counts = q2_tickets.groupby("account_id").size().rename("ticket_count").reset_index()
segment_rates = active.merge(ticket_counts, on="account_id", how="left").fillna({"ticket_count": 0})
q2 = str(segment_rates.groupby("segment")["ticket_count"].mean().sort_values(ascending=False).index[0])

plan_history["month"] = pd.to_datetime(plan_history["month"])
tier_rank = {"starter": 0, "growth": 1, "enterprise": 2}
plan_history["rank"] = plan_history["plan_tier"].map(tier_rank)
upgraded = set()
for account_id, group in plan_history.sort_values("month").groupby("account_id"):
    if (group["rank"].diff().fillna(0) > 0).any():
        upgraded.add(account_id)
print(q2)


SMB


Q3: How many accounts both upgraded plan tier at least once and had zero late invoices?

In [5]:
late_count = invoices.groupby("account_id").apply(lambda g: int((g["paid_date"] > g["due_date"]).sum()), include_groups=False)
zero_late = set(late_count[late_count == 0].index)
q3 = str(len(upgraded & zero_late))
print(q3)


2


Q4: What is the difference between the mean net invoice amount for upgraded accounts and the mean net invoice amount for never-upgraded accounts, rounded to 2 decimals?

In [6]:
account_mean_net = invoices.groupby("account_id")["net_amount"].mean()
diff = account_mean_net[account_mean_net.index.isin(upgraded)].mean() - account_mean_net[~account_mean_net.index.isin(upgraded)].mean()
q4 = f"{diff:.2f}"
print(q4)

-280.50
